Data Preprocessing and standardization

In [9]:
import os
import torch
import torchvision
import sklearn
import pandas as pd
import cv2
from torch.utils.data import Dataset
from PIL import Image
import warnings
warnings.filterwarnings("ignore")
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [10]:
base_image_dir = './data/train_images/'

df = pd.read_csv('./data/train.csv')


# create a new column 'filename' that matches actual file on disk.
df['filename'] = df['id_code'].astype(str) + '.png'

#reduce the number of classes from 5 to 3 for better class balance
def reduce_classes(label):
    if label == 0: return 0        # Normal
    elif label <= 2: return 1      # Mild/Moderate
    else: return 2                 # Severe/Proliferative
df['diagnosis_3cat'] = df['diagnosis'].apply(reduce_classes)


print(df.head())

        id_code  diagnosis          filename  diagnosis_3cat
0  000c1434d8d7          2  000c1434d8d7.png               1
1  001639a390f0          4  001639a390f0.png               2
2  0024cdab0c1e          1  0024cdab0c1e.png               1
3  002c21358ce6          0  002c21358ce6.png               0
4  005b95c28852          0  005b95c28852.png               0


In [11]:
import numpy as np

def crop_image_from_gray(img, tol=7):
    """
    Crops the black borders using a threshold (tol).
    if the image is too dark, it returns the original.
    """
    if img.ndim == 2:
        mask = img > tol
        return img[np.ix_(mask.any(1), mask.any(0))]
    elif img.ndim == 3:
        gray_img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        mask = gray_img > tol
        
        check_shape = img[:,:,0][np.ix_(mask.any(1), mask.any(0))].shape[0]
        if (check_shape == 0): # Image is too dark, return original
            return img
        else:
            img1 = img[:,:,0][np.ix_(mask.any(1), mask.any(0))]
            img2 = img[:,:,1][np.ix_(mask.any(1), mask.any(0))]
            img3 = img[:,:,2][np.ix_(mask.any(1), mask.any(0))]
            img = np.stack([img1, img2, img3], axis=-1)
    return img

def circle_crop(img, sigmaX=10):
    """
    Step 1: Crop black borders
    Step 2: Resize to standard size (e.g., 224 or 256)
    Step 3: Apply Ben Graham's preprocessing method
    """
    # removing black borders
    img = crop_image_from_gray(img)
    
    # resizing to standard size
    img = cv2.resize(img, (224, 224))
    
    # Ben Graham's Method (Color + Gaussian Blur)
    # formula balances the brightness to a middle gray (128)
    # and subtracts the "local average" color (blur) to highlight edges.
    img = cv2.addWeighted(img, 4, cv2.GaussianBlur(img, (0,0), sigmaX), -4, 128)
    
    return img

In [16]:

class RetinopathyDataset(Dataset):
    def __init__(self, dataframe, root_dir, transform=None):
        self.df = dataframe
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = os.path.join(self.root_dir, self.df.iloc[idx]['filename'])
        
        # Open with OpenCV for the preprocessing
        image = cv2.imread(img_name)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # Fix color fix
        
        # applying ben graham's preprocessing
        image = circle_crop(image) 
        
        # convert to PIL for PyTorch transforms compatibility   
        image = Image.fromarray(image)

        if self.transform:
            image = self.transform(image)
        
        label = int(self.df.iloc[idx]['diagnosis_3cat'])
        return image, label

Now we start with Transfer training

Here  we are using transfer learning and the model ResNet50

In [17]:
import torch.nn as nn
import torchvision.models as models
from torch.utils.data import DataLoader
import torchvision.transforms as transforms


In [ ]:
# few more transformations before training 

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [19]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df['diagnosis_3cat'],
    random_state=42
)

train_dataset = RetinopathyDataset(
    train_df,
    base_image_dir,
    transform=train_transform
)

val_dataset = RetinopathyDataset(
    val_df,
    base_image_dir,
    transform=val_transform
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)


In [20]:
model = models.resnet50(pretrained=True)

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, 3)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to C:\Users\HP/.cache\torch\hub\checkpoints\resnet50-0676ba61.pth


100.0%


In [21]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)


In [22]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-4)


In [23]:
def train(model, train_loader, val_loader, epochs=5):
    for epoch in range(epochs):
        model.train()
        train_loss = 0

        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        print(f"Epoch {epoch+1} | Loss: {train_loss/len(train_loader):.4f}")


In [24]:
train(model, train_loader, val_loader, epochs=10)

Epoch 1 | Loss: 0.8064
Epoch 2 | Loss: 0.6177
Epoch 3 | Loss: 0.5490
Epoch 4 | Loss: 0.5093
Epoch 5 | Loss: 0.4893
Epoch 6 | Loss: 0.4686
Epoch 7 | Loss: 0.4598
Epoch 8 | Loss: 0.4463
Epoch 9 | Loss: 0.4509
Epoch 10 | Loss: 0.4261


In [25]:
torch.save(model.state_dict(), "resnet50_dr.pth")

testing the model


In [26]:
model.eval()  # put model in test mode

correct = 0
total = 0

with torch.no_grad():  # no learning, only checking
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Validation Accuracy: {accuracy:.2f}%")


Validation Accuracy: 82.54%


In [31]:
# for storing all the predictions and the labels for the confusion matrix below

from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())


confusion matrix


In [28]:
cm = confusion_matrix(all_labels, all_preds)
print("Confusion Matrix:")
print(cm)


Confusion Matrix:
[[343  17   1]
 [ 24 229  21]
 [  5  60  33]]


final report

In [29]:
print("Classification Report:")
print(classification_report(all_labels, all_preds, target_names=[
    "Normal",
    "Mild/Moderate",
    "Severe"
]))


Classification Report:
               precision    recall  f1-score   support

       Normal       0.92      0.95      0.94       361
Mild/Moderate       0.75      0.84      0.79       274
       Severe       0.60      0.34      0.43        98

     accuracy                           0.83       733
    macro avg       0.76      0.71      0.72       733
 weighted avg       0.81      0.83      0.81       733

